In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import hashlib, urllib.request, zipfile, subprocess, sys, importlib.util

WORK = Path("/content/drive/MyDrive/MiFO")
for sub in ["data/raw/fakenewsnet", "data/raw/liar", "data/processed/step3_pilot", "data/processed/step3_full"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

print("1. Downloading FakeNewsNet CSVs to Drive...")
BASE = "https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset"
for name in ["politifact_fake.csv", "politifact_real.csv", "gossipcop_fake.csv", "gossipcop_real.csv"]:
    dest = WORK / "data/raw/fakenewsnet" / name
    if not dest.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}", dest)
        print(f"   Downloaded {name}")
    else:
        print(f"   Already exists: {name}")

print("\n2. Downloading LIAR dataset to Drive...")
zpath = WORK / "data/raw/liar_dataset.zip"
if not (WORK / "data/raw/liar/train.tsv").exists():
    if not zpath.exists():
        try:
            urllib.request.urlretrieve("https://www.cs.ucsb.edu/~william/data/liar_dataset.zip", zpath)
        except Exception:
            # Fallback mirror if UCSB blocks
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/train.tsv", WORK / "data/raw/liar/train.tsv")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/test.tsv", WORK / "data/raw/liar/test.tsv")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/val.tsv", WORK / "data/raw/liar/valid.tsv")
    if zpath.exists() and not (WORK / "data/raw/liar/train.tsv").exists():
        with zipfile.ZipFile(zpath) as z:
            z.extractall(WORK / "data/raw/liar")
    print("   LIAR ready!")
else:
    print("   LIAR already exists!")

print("\n3. Installing dependencies...")
for pkg in ["sklearn", "trafilatura"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\nSUCCESS! WORK =", WORK)
print("Data is now permanently saved in your Google Drive.")

1. Downloading FakeNewsNet CSVs to Drive...
   Already exists: politifact_fake.csv
   Already exists: politifact_real.csv
   Already exists: gossipcop_fake.csv
   Already exists: gossipcop_real.csv

2. Downloading LIAR dataset to Drive...
   LIAR already exists!

3. Installing dependencies...

SUCCESS! WORK = /content/drive/MyDrive/MiFO
Data is now permanently saved in your Google Drive.


In [3]:
import importlib.util, subprocess, sys
from pathlib import Path

print("platform:", sys.platform, "| cwd:", Path.cwd())
print("'C:/Users/anush/MiFO' exists:", Path("C:/Users/anush/MiFO").exists())

WORK = None
for cand in [Path("C:/Users/anush/MiFO"), Path("/mnt/c/Users/anush/MiFO"),
             Path("/content/drive/MyDrive/MiFO")]:
    if (cand / "data/raw/fakenewsnet/politifact_fake.csv").exists():
        WORK = cand; break
assert WORK is not None, "Auto-locate failed — paste the lines above back to me."
print("WORK =", WORK)

for pkg in ["sklearn", "trafilatura"]:
    if importlib.util.find_spec(pkg) is None:
        print("installing", pkg)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
        if r.returncode != 0:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "--break-system-packages", pkg])
print("deps ready")

platform: linux | cwd: /content
'C:/Users/anush/MiFO' exists: False
WORK = /content/drive/MyDrive/MiFO
deps ready


In [4]:
#Cell A1
import pandas as pd, numpy as np
from urllib.parse import urlparse

RAW = WORK / "data/raw/fakenewsnet"
frames = []
for g in ["politifact", "gossipcop"]:
    for l in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{g}_{l}.csv")
        df["source_group"], df["label_name"], df["label"] = g, l, (l == "fake")
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan": return ""
    if not u.startswith(("http://", "https://")): u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net
fn["domain"] = fn["news_url"].map(domain_of)

# minority-mass closure (owed from step 2 — expect ~3,800, near the LOO errors)
d = fn[fn["domain"] != ""].groupby("domain")["label"].agg(n="size", n_fake="sum")
print(f"Minority-label articles across domains: "
      f"{int(np.minimum(d['n_fake'], d['n'] - d['n_fake']).sum())}\n")

d["bucket"] = np.where(d["n_fake"]/d["n"] >= 0.9, "clean_fake",
              np.where(d["n_fake"]/d["n"] <= 0.1, "clean_real", "MIXED"))
fn = fn.merge(d[["bucket"]], left_on="domain", right_index=True, how="left")
fn["bucket"] = fn["bucket"].fillna("no_domain")

pool = fn[(fn["domain"] != "") & (fn["domain"] != "web.archive.org")]
pool = pool.drop_duplicates(subset="news_url")

parts = [pool[pool.source_group == "politifact"]]
for (b, l), s in pool[pool.source_group == "gossipcop"].groupby(["bucket", "label_name"]):
    parts.append(s.sample(min(len(s), 125), random_state=42))
pilot = pd.concat(parts, ignore_index=True)

print(pilot.groupby(["source_group", "bucket", "label_name"]).size().to_string())
print("\nPilot URLs:", len(pilot))
out = WORK / "data/processed/step3_pilot"; out.mkdir(parents=True, exist_ok=True)
pilot.to_csv(out / "pilot_sample.csv", index=False)

Minority-label articles across domains: 3184

source_group  bucket      label_name
gossipcop     MIXED       fake          125
                          real          125
              clean_fake  fake          125
                          real           27
              clean_real  fake          125
                          real          125
politifact    MIXED       fake           57
                          real          117
              clean_fake  fake          290
                          real            1
              clean_real  fake           12
                          real          301

Pilot URLs: 1430


In [3]:
#Cell A2
import requests, time, threading, trafilatura, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

PILOT = pd.read_csv(WORK / "data/processed/step3_pilot/pilot_sample.csv")
TXT_DIR = WORK / "data/raw/crawl_pilot"; TXT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = WORK / "data/processed/step3_pilot/results.csv"

# EDIT: put your real email here — honest crawlers identify themselves
UA_RESEARCH = {"User-Agent": "MiFOResearchBot/0.1 (academic study; contact: YOU@EMAIL.COM)"}
UA_BROWSER = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

domain_last, lock = defaultdict(float), threading.Lock()
MIN_GAP = 2.0  # seconds between hits to the same domain

def polite_wait(dom):
    while True:
        with lock:
            if time.time() >= domain_last[dom] + MIN_GAP:
                domain_last[dom] = time.time(); return
        time.sleep(0.5)

def fetch(url):
    url = str(url).strip()
    if not url.startswith(("http://", "https://")): url = "http://" + url
    try:
        r = requests.get(url, headers=UA_RESEARCH, timeout=15, allow_redirects=True)
        if r.status_code in (403, 429, 503):   # one polite retry with browser UA
            time.sleep(3)
            r = requests.get(url, headers=UA_BROWSER, timeout=15, allow_redirects=True)
        return r
    except Exception as e:
        return None

def crawl_row(row):
    polite_wait(row["domain"])
    r = fetch(row["news_url"])
    rec = {"id": row["id"], "group": row["source_group"], "label": row["label_name"],
           "bucket": row["bucket"], "url": row["news_url"], "status": None,
           "final_url": None, "text_len": 0}
    if r is not None:
        rec["status"], rec["final_url"] = r.status_code, r.url
        if r.status_code == 200 and "html" in r.headers.get("Content-Type", "").lower():
            text = trafilatura.extract(r.text, include_comments=False,
                                       include_tables=False) or ""
            rec["text_len"] = len(text)
            if text:
                (TXT_DIR / f"{row['id']}.txt").write_text(text, encoding="utf-8")
    else:
        rec["status"] = "error"
    return rec

rows, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    futs = [ex.submit(crawl_row, r) for r in PILOT.to_dict("records")]
    for i, f in enumerate(as_completed(futs), 1):
        rows.append(f.result())
        if i % 100 == 0:
            pd.DataFrame(rows).to_csv(RESULTS, index=False)
            print(f"{i}/{len(PILOT)} done ({time.time()-t0:.0f}s)")
pd.DataFrame(rows).to_csv(RESULTS, index=False)
print(f"FINISHED {len(rows)} in {(time.time()-t0)/60:.1f} min -> {RESULTS}")

100/1430 done (18s)


200/1430 done (39s)


300/1430 done (59s)


400/1430 done (76s)
500/1430 done (95s)
600/1430 done (114s)
700/1430 done (135s)


800/1430 done (160s)


...

1000/1430 done (208s)


1100/1430 done (236s)


1200/1430 done (276s)


ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None


1300/1430 done (305s)
1400/1430 done (325s)
FINISHED 1430 in 5.7 min -> /content/drive/MyDrive/MiFO/data/processed/step3_pilot/results.csv


In [4]:
#Cell C2
import pandas as pd, numpy as np
COLS = ["json_id","label","statement","subjects","speaker","job","state","party",
        "barely_true","false","half_true","mostly_true","pants_fire","context"]
liar = {s: pd.read_csv(WORK / f"data/raw/liar/{s}.tsv", sep="\t",
                       names=COLS, quoting=3) for s in ["train","valid","test"]}
tr = liar["train"].copy()
tr["words"] = tr["statement"].str.split().str.len()

print("=== Statement length (words) by class ===")
print(tr.groupby("label")["words"].agg(["count","median","mean"]).round(1)
      .sort_values("median").to_string())

print(f"\n=== Speakers: {tr['speaker'].nunique()} unique; top 5 ===")
print(tr["speaker"].value_counts().head(5).to_string())

print("\n=== Credit history (mean prior counts) by current class ===")
hist = ["barely_true","false","half_true","mostly_true","pants_fire"]
h = tr[hist].apply(pd.to_numeric, errors="coerce")
h["label"] = tr["label"].values
print(h.groupby("label").mean().round(1).to_string())

subj = tr["subjects"].fillna("").str.split(",").explode().str.strip()
print("\n=== Top subjects ===");  print(subj[subj != ""].value_counts().head(10).to_string())

=== Statement length (words) by class ===
             count  median  mean
label                           
false         1998    15.0  16.8
pants-fire     842    16.0  17.1
true          1683    16.0  17.9
barely-true   1657    17.0  18.1
mostly-true   1966    17.0  18.2
half-true     2123    18.0  18.8

=== Speakers: 2916 unique; top 5 ===
speaker
barack-obama       493
donald-trump       274
hillary-clinton    239
mitt-romney        180
scott-walker       150

=== Credit history (mean prior counts) by current class ===
             barely_true  false  half_true  mostly_true  pants_fire
label                                                              
barely-true         11.7   12.4       14.8         13.9         5.4
false               11.7   15.9       15.2         14.0         7.9
half-true           11.9   12.6       19.6         18.2         4.5
mostly-true         11.8   12.3       19.8         20.6         3.8
pants-fire          11.0   18.1       11.5          9.2        1

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

pipe = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
                     LogisticRegression(max_iter=2000))
pipe.fit(liar["train"]["statement"], liar["train"]["label"])
pred = pipe.predict(liar["test"]["statement"])

maj = liar["train"]["label"].value_counts(normalize=True).iloc[0]
acc = accuracy_score(liar["test"]["label"], pred)
f1m = f1_score(liar["test"]["label"], pred, average="macro")
print(f"majority baseline: {maj:.3f}  |  TF-IDF+LR accuracy: {acc:.3f}  |  macro-F1: {f1m:.3f}\n")
print(classification_report(liar["test"]["label"], pred, digits=2))

out = WORK / "data/processed/step3_liar"; out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"y": liar["test"]["label"], "pred": pred}).to_csv(out / "lr_baseline_preds.csv", index=False)
print("saved ->", out / "lr_baseline_preds.csv")

majority baseline: 0.207  |  TF-IDF+LR accuracy: 0.252  |  macro-F1: 0.219

              precision    recall  f1-score   support

 barely-true       0.23      0.14      0.18       214
       false       0.31      0.40      0.35       250
   half-true       0.22      0.29      0.25       267
 mostly-true       0.23      0.27      0.25       249
  pants-fire       0.33      0.03      0.06        92
        true       0.25      0.21      0.23       211

    accuracy                           0.25      1283
   macro avg       0.26      0.22      0.22      1283
weighted avg       0.25      0.25      0.24      1283

saved -> /content/drive/MyDrive/MiFO/data/processed/step3_liar/lr_baseline_preds.csv


In [6]:
#Cell A3
res = pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
res["live"] = res["status"] == 200
res["extracted"] = res["text_len"] >= 200

print(f"HTTP 200: {res['live'].mean():.1%}  |  extracted(>=200 chars): "
      f"{res['extracted'].mean():.1%}  |  median text len: "
      f"{res.loc[res.extracted, 'text_len'].median():.0f}")

print("\n=== By group/label ===")
print(res.groupby(["group","label"]).agg(n=("id","size"), live=("live","mean"),
      extracted=("extracted","mean")).round(3).to_string())

print("\n=== By bucket ===")
print(res.groupby("bucket").agg(n=("id","size"), live=("live","mean"),
      extracted=("extracted","mean")).round(3).to_string())

print("\n=== Status codes ===")
print(res["status"].value_counts(dropna=False).head(10).to_string())

HTTP 200: 0.0%  |  extracted(>=200 chars): 44.7%  |  median text len: 2500

=== By group/label ===
                    n  live  extracted
group      label                      
gossipcop  fake   375   0.0      0.619
           real   277   0.0      0.632
politifact fake   359   0.0      0.298
           real   419   0.0      0.298

=== By bucket ===
              n  live  extracted
bucket                          
MIXED       424   0.0      0.561
clean_fake  443   0.0      0.368
clean_real  563   0.0      0.423

=== Status codes ===
status
200      773
error    199
403      174
404      173
402       51
202       15
400       14
410       11
500        6
503        5


In [7]:
#extra A3
res["live"] = pd.to_numeric(res["status"], errors="coerce") == 200
print("Real HTTP 200 Live Rate:", res["live"].mean())

Real HTTP 200 Live Rate: 0.5405594405594406


In [8]:
import requests, pandas as pd, time

res = pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
failed = res[(pd.to_numeric(res["status"], errors="coerce") != 200)
             | (res["text_len"] < 200)].copy()
sample = failed.sample(min(20, len(failed)), random_state=42)

for i, (_, r) in enumerate(sample.iterrows(), 1):
    url = str(r["url"]).strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    try:
        resp = requests.get("https://archive.org/wayback/available",
                            params={"url": url}, timeout=20)
        print(f"{i:2d}  {resp.status_code}  {resp.text[:150]}")
    except Exception as e:
        print(f"{i:2d}  EXCEPTION {type(e).__name__}: {str(e)[:80]}")
    time.sleep(2)

 1  200  {"url": "https://www.stlouisfed.org/~/media/Files/PDFs/publications/pub_assets/pdf/itb/2014/In%20the%20Balance%20June%20issue%208.pdf", "archived_snap
 2  200  {"url": "http://worldnewsdailyreport.com/bin-laden-is-alive-and-well-in-the-bahamas-says-edward-snowden/", "archived_snapshots": {"closest": {"status"
 3  200  {"url": "http://www.whitehouse.gov/the-press-office/2012/08/09/remarks-president-campaign-event-colorado-springs-co", "archived_snapshots": {"closest"
 4  200  {"url": "http://www.cbsnews.com/htdocs/pdf/FTN_083108.pdf", "archived_snapshots": {"closest": {"status": "200", "available": true, "url": "http://web.
 5  200  {"url": "http://thomas.loc.gov/cgi-bin/query/D?c108:5:./temp/%7Ec108Feg4Je::", "archived_snapshots": {}}
 6  200  {"url": "https://hollywoodlife.com/2018/04/27/chance-the-rapper-apologizes-kanye-west-donald-trump-comment-democrat-tweet/", "archived_snapshots": {}}
 7  200  {"url": "http://www.foxnews.com/story/0,2933,183845,00.html", "archived_snaps

In [9]:
import pandas as pd
from urllib.parse import urlparse

RAW = WORK / "data/raw/fakenewsnet"
frames = []
for g in ["politifact", "gossipcop"]:
    for l in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{g}_{l}.csv")
        df["source_group"], df["label_name"], df["label"] = g, l, (l == "fake")
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan": return ""
    if not u.startswith(("http://", "https://")): u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net
fn["domain"] = fn["news_url"].map(domain_of)

pool = fn[(fn["domain"] != "") & (fn["domain"] != "web.archive.org")]
pool = pool.drop_duplicates(subset="news_url")

pilot_done = set(pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
                 .query("text_len >= 200")["id"])
manifest = pool[~pool["id"].isin(pilot_done)].copy()

out = WORK / "data/processed/step3_full"; out.mkdir(parents=True, exist_ok=True)
manifest.to_csv(out / "manifest.csv", index=False)
print(f"unique URLs: {len(pool)} | already have text (pilot): {len(pilot_done)}")
print(f"TO CRAWL: {len(manifest)}")
print(manifest.groupby(["source_group", "label_name"]).size().to_string())

unique URLs: 21461 | already have text (pilot): 639
TO CRAWL: 20822
source_group  label_name
gossipcop     fake           4449
              real          15827
politifact    fake            252
              real            294


In [10]:
# F5.1 — Stable Batch-Fed Full Crawl (Crash-Proof)
import subprocess, time, threading, trafilatura, pandas as pd, zipfile, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from pathlib import Path

UA_EMAIL = "anushaanand1543@gmail.com"
WORK  = Path("/content/drive/MyDrive/MiFO")
MAN   = pd.read_csv(WORK / "data/processed/step3_full/manifest.csv")
RES_D = WORK / "data/processed/step3_full/results.csv"
RES_L = Path("/content/results_local.csv")
TXT_L = Path("/content/crawl_full"); TXT_L.mkdir(exist_ok=True)
TXT_D = WORK / "data/raw/crawl_full"
ZIP_D = WORK / "data/processed/step3_full/texts_snapshot.zip"

UA_R = f"MiFOResearchBot/0.1 (academic study; contact: {UA_EMAIL})"
UA_B = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"

# 1. Recover any existing texts from local & Drive
rescued = {p.stem: p.stat().st_size for p in TXT_L.glob("*.txt")}
if TXT_D.exists():
    for p in TXT_D.glob("*.txt"):
        dest = TXT_L / p.name
        if not dest.exists():
            dest.write_bytes(p.read_bytes())
            rescued[p.stem] = dest.stat().st_size
print(f"Texts already saved on local disk: {len(rescued)}")

# 2. Check completed IDs
frames = []
if RES_L.exists(): frames.append(pd.read_csv(RES_L))
if RES_D.exists(): frames.append(pd.read_csv(RES_D))
done_ids, rec_rows = set(), []
if frames:
    prev = pd.concat(frames, ignore_index=True).drop_duplicates(subset="id", keep="last")
    done_ids |= set(prev["id"]); rec_rows.append(prev)

back = MAN[MAN["id"].isin(set(rescued) - done_ids)]
if len(back):
    rec_rows.append(pd.DataFrame({
        "id": back["id"], "group": back["source_group"], "label": back["label_name"],
        "bucket": back.get("bucket", "unknown"), "url": back["news_url"], "status": 200,
        "final_url": None, "text_len": [rescued[i] for i in back["id"]]}))
    done_ids |= set(back["id"])
if rec_rows:
    pd.concat(rec_rows, ignore_index=True).to_csv(RES_L, index=False)

todo = MAN[~MAN["id"].isin(done_ids)].sample(frac=1, random_state=42)
print(f"Manifest: {len(MAN)} | Already Done: {len(done_ids)} | Remaining Todo: {len(todo)}")

# 3. cURL fetcher (10s hard max)
domain_last, lock = defaultdict(float), threading.Lock()
MIN_GAP = 1.0

def polite_wait(dom):
    while True:
        with lock:
            if time.time() >= domain_last[dom] + MIN_GAP:
                domain_last[dom] = time.time(); return
        time.sleep(0.3)

def _curl_get(url, ua, max_time=10):
    try:
        r = subprocess.run(
            ["curl", "-s", "-L", "--compressed", "--max-time", str(max_time),
             "-A", ua, "-w", "\n__MIFO__%{http_code}", url],
            capture_output=True, text=True, errors="replace", timeout=max_time + 3)
        if r.returncode != 0 or "__MIFO__" not in r.stdout:
            return None
        body, code = r.stdout.rsplit("\n__MIFO__", 1)
        class _R: pass
        resp = _R()
        resp.status_code = int(code) if code.isdigit() else 0
        resp.text = body
        resp.headers = {"Content-Type": "text/html" if body.lstrip()[:1] == "<" else "other"}
        return resp
    except Exception:
        return None

def fetch(url):
    url = str(url).strip()
    if not url.startswith(("http://", "https://")): url = "http://" + url
    r = _curl_get(url, UA_R)
    if r is not None and r.status_code in (403, 429, 503):
        time.sleep(2)
        r = _curl_get(url, UA_B)
    return r

def crawl_row(row):
    polite_wait(row.get("domain", ""))
    r = fetch(row["news_url"])
    rec = {"id": row["id"], "group": row["source_group"], "label": row["label_name"],
           "bucket": row.get("bucket", "unknown"), "url": row["news_url"], "status": None,
           "final_url": None, "text_len": 0}
    if r is not None:
        rec["status"], rec["final_url"] = r.status_code, row["news_url"]
        if r.status_code == 200 and "html" in r.headers.get("Content-Type", "").lower():
            try:
                text = trafilatura.extract(r.text, include_comments=False, include_tables=False) or ""
            except Exception:
                text = ""
            rec["text_len"] = len(text)
            if text:
                (TXT_L / f"{row['id']}.txt").write_text(text, encoding="utf-8")
    else:
        rec["status"] = "error"
    return rec

# 4. Heartbeat
stats = {"done": len(done_ids), "texts": len(rescued), "t0": time.time()}
stop = threading.Event()
def heartbeat():
    time.sleep(30)
    while not stop.is_set():
        el = time.time() - stats["t0"]
        print(f"  [hb {el/60:4.1f}m] progress={stats['done']}/{len(MAN)} "
              f"texts={stats['texts']} ({stats['done']/max(1, el):.1f}/s)", flush=True)
        time.sleep(60)
threading.Thread(target=heartbeat, daemon=True).start()

def sync_to_drive():
    if RES_L.exists():
        pd.read_csv(RES_L).to_csv(RES_D, index=False)
    zp = "/content/texts_snapshot.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in TXT_L.glob("*.txt"):
            z.write(p, arcname=p.name)
    shutil.copy(zp, ZIP_D)
    print(f"  [sync] Snapshot saved to Drive", flush=True)

# 5. Safe Batch Execution (12 workers, in chunks of 300)
BATCH_SIZE = 300
all_records = todo.to_dict("records")

for start_idx in range(0, len(all_records), BATCH_SIZE):
    batch = all_records[start_idx : start_idx + BATCH_SIZE]
    rows = []
    with ThreadPoolExecutor(max_workers=12) as ex:
        futs = [ex.submit(crawl_row, r) for r in batch]
        for f in as_completed(futs):
            rec = f.result()
            rows.append(rec)
            stats["done"] += 1
            if rec["text_len"]: stats["texts"] += 1
    
    # Save after each batch
    done = pd.concat([pd.read_csv(RES_L), pd.DataFrame(rows)], ignore_index=True) if RES_L.exists() else pd.DataFrame(rows)
    done.drop_duplicates(subset="id", keep="last").to_csv(RES_L, index=False)
    
    if stats["done"] % 1500 < BATCH_SIZE:
        sync_to_drive()

stop.set()
sync_to_drive()

# 6. Final Status
res = pd.read_csv(RES_L if RES_L.exists() else RES_D)
res["ok"] = pd.to_numeric(res["status"], errors="coerce") == 200
print("\nFINISHED CRAWL!")
print(res.groupby(["group", "label"]).agg(n=("id", "size"), live=("ok", "mean"),
      with_text=("text_len", lambda s: (s >= 200).mean())).round(3).to_string())
print("\nTOTALS:", len(res), "attempted |", int((res["text_len"] >= 200).sum()), "with text")

Texts already saved on local disk: 0
Manifest: 20822 | Already Done: 20810 | Remaining Todo: 22
  [sync] Snapshot saved to Drive

FINISHED CRAWL!
                      n   live  with_text
group      label                         
gossipcop  fake    4451  0.604      0.567
           real   15829  0.625      0.602
politifact fake     253  0.103      0.008
           real     299  0.351      0.037

TOTALS: 20832 attempted | 12063 with text


In [6]:
# W1 — Resumable Wayback CDX Resolver (Syncs to Google Drive continuously)
import subprocess, time, threading, json, pandas as pd, shutil
from urllib.parse import urlparse, quote
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

WORK = Path("/content/drive/MyDrive/MiFO")
UA_R = "MiFOResearchBot/0.1 (academic study; contact: anushaanand1543@gmail.com)"

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan": return ""
    if not u.startswith(("http://", "https://")): u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net

frames = []
for g in ["politifact", "gossipcop"]:
    for l in ["fake", "real"]:
        df = pd.read_csv(WORK / f"data/raw/fakenewsnet/{g}_{l}.csv")
        df["source_group"], df["label_name"] = g, l
        frames.append(df)
fn = pd.concat(frames, ignore_index=True).drop_duplicates(subset="news_url")
fn["domain"] = fn["news_url"].map(domain_of)

have = set()
for f in ["data/processed/step3_full/results.csv", "data/processed/step3_pilot/results.csv"]:
    p = WORK / f
    if p.exists():
        r = pd.read_csv(p)
        have |= set(r.loc[r["text_len"] >= 200, "id"])
print(f"Articles with text already: {len(have)}")

failed = fn[~fn["id"].isin(have) & (fn["domain"] != "")].copy()
failed["is_archive_url"] = failed["domain"] == "web.archive.org"
failed = failed.sort_values("source_group", key=lambda s: (s != "politifact"))  # politifact first
print(f"Wayback targets: {len(failed)} (archive-URL rows: {failed['is_archive_url'].sum()})")

WIDX_L = Path("/content/wayback_index.csv")
WIDX_D = WORK / "data/processed/step3_full/wayback_index.csv"
WIDX_D.parent.mkdir(parents=True, exist_ok=True)

# 1. Check BOTH Google Drive and local disk for completed IDs
done_frames = []
if WIDX_D.exists():
    done_frames.append(pd.read_csv(WIDX_D))
if WIDX_L.exists():
    done_frames.append(pd.read_csv(WIDX_L))

done_ids = set()
if done_frames:
    master_done = pd.concat(done_frames, ignore_index=True).drop_duplicates(subset="id", keep="last")
    master_done.to_csv(WIDX_L, index=False)
    master_done.to_csv(WIDX_D, index=False)
    done_ids = set(master_done["id"])

todo = failed[~failed["id"].isin(done_ids)]
print(f"Previously Done: {len(done_ids)} | Remaining Todo: {len(todo)}")

def _curl_get(url, ua, max_time=25):
    try:
        r = subprocess.run(["curl", "-s", "--max-time", str(max_time), "-A", ua,
                            "-w", "\n__M__%{http_code}", url],
                           capture_output=True, text=True, errors="replace", timeout=max_time + 5)
        if r.returncode != 0 or "__M__" not in r.stdout: return None
        body, code = r.stdout.rsplit("\n__M__", 1)
        return (int(code) if code.isdigit() else 0, body)
    except Exception:
        return None

lock, wb_last = threading.Lock(), {"t": 0.0}
def wb_wait():
    while True:
        with lock:
            if time.time() >= wb_last["t"] + 0.7:
                wb_last["t"] = time.time(); return
        time.sleep(0.2)

def cdx(row):
    if row["is_archive_url"]:
        return {"id": row["id"], "status": "direct_archive", "ts": None, "url": row["news_url"]}
    q = ("https://web.archive.org/cdx/search/cdx?url=" + quote(str(row["news_url"]), safe="")
         + "&output=json&filter=statuscode:200&fl=timestamp&from=2015&to=2019&collapse=digest&limit=1")
    wb_wait()
    r = _curl_get(q, UA_R)
    if r is None or r[0] != 200: st = "cdx_error"
    else:
        try:
            rows = json.loads(r[1]); st, ts = ("ok", rows[1][0]) if len(rows) >= 2 else ("no_snapshot", None)
        except Exception: st, ts = "cdx_error", None
    return {"id": row["id"], "status": st, "ts": ts if st == "ok" else None,
            "url": row["news_url"]}

stats = {"n": len(done_ids), "ok": 0}
t0 = time.time()
stop = threading.Event()

def hb():
    while not stop.is_set():
        time.sleep(60)
        el = time.time() - t0
        print(f"  [hb {el/60:4.1f}m] progress={stats['n']}/{len(failed)} snapshots_found={stats['ok']} "
              f"({(stats['n']-len(done_ids))/max(1,el):.1f}/s)", flush=True)
threading.Thread(target=hb, daemon=True).start()

def sync_index_to_drive():
    if WIDX_L.exists():
        shutil.copy(WIDX_L, WIDX_D)
        print(f"  [sync] Saved wayback_index.csv to Google Drive", flush=True)

new = []
with ThreadPoolExecutor(max_workers=2) as ex:
    for rec in ex.map(cdx, todo.to_dict("records")):
        new.append(rec)
        stats["n"] += 1
        stats["ok"] += (rec["status"] == "ok")
        if len(new) % 200 == 0:
            df = pd.concat([pd.read_csv(WIDX_L), pd.DataFrame(new)], ignore_index=True) if WIDX_L.exists() else pd.DataFrame(new)
            df.drop_duplicates(subset="id", keep="last").to_csv(WIDX_L, index=False)
            new = []
            sync_index_to_drive()

if new:
    df = pd.concat([pd.read_csv(WIDX_L), pd.DataFrame(new)], ignore_index=True) if WIDX_L.exists() else pd.DataFrame(new)
    df.drop_duplicates(subset="id", keep="last").to_csv(WIDX_L, index=False)
    sync_index_to_drive()

stop.set()

idx = pd.read_csv(WIDX_D)
print("\nFINISHED WAYBACK RESOLUTION!")
print(idx["status"].value_counts().to_string())

Articles with text already: 12696
Wayback targets: 8962 (archive-URL rows: 197)
Previously Done: 7200 | Remaining Todo: 1762
  [hb  0.0m] progress=7201/8962 snapshots_found=0 (1.0/s)
  [hb  0.9m] progress=7258/8962 snapshots_found=2 (1.1/s)
  [hb  1.0m] progress=7262/8962 snapshots_found=3 (1.0/s)
  [hb  1.0m] progress=7262/8962 snapshots_found=3 (1.0/s)
  [hb  1.9m] progress=7330/8962 snapshots_found=5 (1.2/s)
  [hb  2.0m] progress=7342/8962 snapshots_found=5 (1.2/s)
  [hb  2.0m] progress=7342/8962 snapshots_found=5 (1.2/s)
  [sync] Saved wayback_index.csv to Google Drive
  [hb  2.9m] progress=7409/8962 snapshots_found=6 (1.2/s)
  [hb  3.0m] progress=7421/8962 snapshots_found=6 (1.2/s)
  [hb  3.0m] progress=7421/8962 snapshots_found=6 (1.2/s)
  [hb  3.9m] progress=7490/8962 snapshots_found=8 (1.3/s)
  [hb  4.0m] progress=7500/8962 snapshots_found=9 (1.2/s)
  [hb  4.0m] progress=7500/8962 snapshots_found=9 (1.2/s)
  [hb  4.9m] progress=7557/8962 snapshots_found=11 (1.2/s)
  [hb  5.0m] 

/tmp/ipykernel_82324/3917461201.py:118: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(WIDX_L), pd.DataFrame(new)], ignore_index=True) if WIDX_L.exists() else pd.DataFrame(new)


  [hb 17.9m] progress=8448/8962 snapshots_found=28 (1.2/s)
  [hb 18.0m] progress=8462/8962 snapshots_found=28 (1.2/s)
  [hb 18.0m] progress=8462/8962 snapshots_found=28 (1.2/s)
  [hb 18.9m] progress=8509/8962 snapshots_found=28 (1.2/s)
  [hb 19.0m] progress=8509/8962 snapshots_found=28 (1.1/s)
  [hb 19.0m] progress=8509/8962 snapshots_found=28 (1.1/s)
  [sync] Saved wayback_index.csv to Google Drive


/tmp/ipykernel_82324/3917461201.py:118: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(WIDX_L), pd.DataFrame(new)], ignore_index=True) if WIDX_L.exists() else pd.DataFrame(new)


  [hb 19.9m] progress=8600/8962 snapshots_found=28 (1.2/s)
  [hb 20.0m] progress=8612/8962 snapshots_found=28 (1.2/s)
  [hb 20.0m] progress=8612/8962 snapshots_found=28 (1.2/s)
  [hb 20.9m] progress=8671/8962 snapshots_found=28 (1.2/s)
  [hb 21.0m] progress=8675/8962 snapshots_found=28 (1.2/s)
  [hb 21.0m] progress=8675/8962 snapshots_found=28 (1.2/s)
  [hb 21.9m] progress=8747/8962 snapshots_found=28 (1.2/s)
  [hb 22.0m] progress=8759/8962 snapshots_found=28 (1.2/s)
  [hb 22.0m] progress=8759/8962 snapshots_found=28 (1.2/s)
  [sync] Saved wayback_index.csv to Google Drive


/tmp/ipykernel_82324/3917461201.py:118: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(WIDX_L), pd.DataFrame(new)], ignore_index=True) if WIDX_L.exists() else pd.DataFrame(new)


  [hb 22.9m] progress=8827/8962 snapshots_found=28 (1.2/s)
  [hb 23.0m] progress=8839/8962 snapshots_found=28 (1.2/s)
  [hb 23.0m] progress=8839/8962 snapshots_found=28 (1.2/s)
  [hb 23.9m] progress=8901/8962 snapshots_found=28 (1.2/s)
  [hb 24.0m] progress=8911/8962 snapshots_found=28 (1.2/s)
  [hb 24.0m] progress=8911/8962 snapshots_found=28 (1.2/s)
  [sync] Saved wayback_index.csv to Google Drive

FINISHED WAYBACK RESOLUTION!
status
cdx_error         7480
ok                 854
no_snapshot        431
direct_archive     197


/tmp/ipykernel_82324/3917461201.py:124: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(WIDX_L), pd.DataFrame(new)], ignore_index=True) if WIDX_L.exists() else pd.DataFrame(new)


  [hb 24.9m] progress=8962/8962 snapshots_found=28 (1.2/s)
  [hb 25.0m] progress=8962/8962 snapshots_found=28 (1.2/s)
  [hb 25.0m] progress=8962/8962 snapshots_found=28 (1.2/s)


In [7]:
# W2 — Fetch the Resolved Snapshots & Extract Text (Resumable & Safe)
import subprocess, time, threading, trafilatura, pandas as pd, zipfile, shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

WORK = Path("/content/drive/MyDrive/MiFO")
UA_R = "MiFOResearchBot/0.1 (academic study; contact: anushaanand1543@gmail.com)"

idx = pd.read_csv(WORK / "data/processed/step3_full/wayback_index.csv")
targets = idx[idx["status"].isin(["ok", "direct_archive"])].copy()

TXT_L = Path("/content/wayback_texts"); TXT_L.mkdir(exist_ok=True)
RES_L = Path("/content/wayback_results.csv")
RES_D = WORK / "data/processed/step3_full/wayback_results.csv"
ZIP_D = WORK / "data/processed/step3_full/wayback_texts.zip"

done_ids = set()
if RES_D.exists():
    done_ids |= set(pd.read_csv(RES_D)["id"])
if RES_L.exists():
    done_ids |= set(pd.read_csv(RES_L)["id"])

todo = targets[~targets["id"].isin(done_ids)]
print(f"Total Targets: {len(targets)} | Already Done: {len(done_ids)} | Todo Now: {len(todo)}")

def _curl_get(url, ua, max_time=30):
    try:
        r = subprocess.run(["curl", "-s", "-L", "--compressed", "--max-time", str(max_time),
                            "-A", ua, "-w", "\n__M__%{http_code}", url],
                           capture_output=True, text=True, errors="replace", timeout=max_time + 5)
        if r.returncode != 0 or "__M__" not in r.stdout: return None
        body, code = r.stdout.rsplit("\n__M__", 1)
        return (int(code) if code.isdigit() else 0, body)
    except Exception:
        return None

lock, wb_last = threading.Lock(), {"t": 0.0}
def wb_wait():
    while True:
        with lock:
            if time.time() >= wb_last["t"] + 0.7:
                wb_last["t"] = time.time(); return
        time.sleep(0.2)

def fetch_snap(row):
    url = (f"https://web.archive.org/web/{row['ts']}id_/{row['url']}"
           if row["status"] == "ok" else row["url"])
    wb_wait()
    r = _curl_get(url, UA_R)
    rec = {"id": row["id"], "url": url, "status": None, "text_len": 0}
    if r is None:
        rec["status"] = "error"
    else:
        rec["status"] = r[0]
        if r[0] == 200 and r[1].lstrip()[:1] == "<":
            try:
                text = trafilatura.extract(r[1], include_comments=False, include_tables=False) or ""
            except Exception:
                text = ""
            rec["text_len"] = len(text)
            if text:
                (TXT_L / f"{row['id']}.txt").write_text(text, encoding="utf-8")
    return rec

stats, t0 = {"n": len(done_ids), "t": 0}, time.time()
stop = threading.Event()

def hb():
    while not stop.is_set():
        time.sleep(60)
        el = time.time() - t0
        print(f"  [hb {el/60:4.1f}m] progress={stats['n']}/{len(targets)} texts_saved={stats['t']} "
              f"({(stats['n']-len(done_ids))/max(1,el):.1f}/s)", flush=True)
threading.Thread(target=hb, daemon=True).start()

def sync_wayback_to_drive():
    if RES_L.exists():
        pd.read_csv(RES_L).to_csv(RES_D, index=False)
    zp = "/content/wayback_texts.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in TXT_L.glob("*.txt"):
            z.write(p, arcname=p.name)
    shutil.copy(zp, ZIP_D)
    print(f"  [sync] Snapshot zip saved to Google Drive", flush=True)

new = []
with ThreadPoolExecutor(max_workers=2) as ex:
    for rec in ex.map(fetch_snap, todo.to_dict("records")):
        new.append(rec)
        stats["n"] += 1
        stats["t"] += (rec["text_len"] >= 200)
        if len(new) % 150 == 0:
            df = pd.concat([pd.read_csv(RES_L), pd.DataFrame(new)], ignore_index=True) if RES_L.exists() else pd.DataFrame(new)
            df.drop_duplicates(subset="id", keep="last").to_csv(RES_L, index=False)
            new = []
            sync_wayback_to_drive()

if new:
    df = pd.concat([pd.read_csv(RES_L), pd.DataFrame(new)], ignore_index=True) if RES_L.exists() else pd.DataFrame(new)
    df.drop_duplicates(subset="id", keep="last").to_csv(RES_L, index=False)
    sync_wayback_to_drive()

stop.set()

df = pd.read_csv(RES_D)
print("\nFINISHED DOWNLOADING WAYBACK SNAPSHOTS!")
print(f"TOTALS: {len(df)} fetched | {int((df['text_len'] >= 200).sum())} articles with full text")

Total Targets: 1051 | Already Done: 0 | Todo Now: 1051
  [hb  1.0m] progress=77/1051 texts_saved=12 (1.3/s)
  [sync] Snapshot zip saved to Google Drive
  [hb  2.0m] progress=158/1051 texts_saved=24 (1.3/s)
  [hb  3.0m] progress=239/1051 texts_saved=36 (1.3/s)
  [sync] Snapshot zip saved to Google Drive
  [hb  4.0m] progress=315/1051 texts_saved=50 (1.3/s)
  [hb  5.0m] progress=393/1051 texts_saved=68 (1.3/s)
  [sync] Snapshot zip saved to Google Drive
  [hb  6.0m] progress=472/1051 texts_saved=88 (1.3/s)
  [hb  7.0m] progress=553/1051 texts_saved=108 (1.3/s)
  [sync] Snapshot zip saved to Google Drive
  [hb  8.0m] progress=631/1051 texts_saved=127 (1.3/s)
  [hb  9.0m] progress=709/1051 texts_saved=145 (1.3/s)
  [sync] Snapshot zip saved to Google Drive
  [hb 10.0m] progress=786/1051 texts_saved=164 (1.3/s)


  [hb 11.0m] progress=865/1051 texts_saved=183 (1.3/s)
  [sync] Snapshot zip saved to Google Drive
  [hb 12.0m] progress=942/1051 texts_saved=203 (1.3/s)
  [hb 13.0m] progress=1020/1051 texts_saved=223 (1.3/s)
  [sync] Snapshot zip saved to Google Drive
  [sync] Snapshot zip saved to Google Drive

FINISHED DOWNLOADING WAYBACK SNAPSHOTS!
TOTALS: 1051 fetched | 232 articles with full text


  [hb 14.0m] progress=1051/1051 texts_saved=232 (1.3/s)


In [8]:
# W3 — Merge everything into the final corpus index and Lock the Dataset
import pandas as pd
from pathlib import Path

WORK = Path("/content/drive/MyDrive/MiFO")

srcs = [("direct_full",  "data/processed/step3_full/results.csv"),
        ("direct_pilot", "data/processed/step3_pilot/results.csv"),
        ("wayback",      "data/processed/step3_full/wayback_results.csv")]

rows = []
for src, p in srcs:
    path = WORK / p
    if path.exists():
        df = pd.read_csv(path)
        df["source"] = src
        rows.append(df.loc[df["text_len"] >= 200, ["id", "text_len", "source"]])

corpus = pd.concat(rows, ignore_index=True).drop_duplicates(subset="id", keep="first")

meta = pd.concat([pd.read_csv(WORK / f"data/raw/fakenewsnet/{g}_{l}.csv").assign(
    source_group=g, label_name=l) for g in ["politifact", "gossipcop"] for l in ["fake", "real"]],
    ignore_index=True)

corpus = corpus.merge(meta[["id", "source_group", "label_name", "title"]], on="id", how="left")

print(f"\n==========================================")
print(f"🎯 CORPUS LOCKED: {len(corpus)} articles with full text ({100*len(corpus)/len(meta):.1f}% of {len(meta)})")
print(f"==========================================")
print(corpus.groupby(["source_group", "label_name"]).size().to_string())
print("\nBy Ingestion Source:")
print(corpus["source"].value_counts().to_string())

out = WORK / "data/processed/corpus_index.csv"
corpus.to_csv(out, index=False)
print(f"\nMaster index saved to -> {out}")


🎯 CORPUS LOCKED: 12930 articles with full text (55.7% of 23196)
source_group  label_name
gossipcop     fake          2766
              real          9836
politifact    fake           159
              real           169

By Ingestion Source:
source
direct_full     12063
direct_pilot      634
wayback           233

Master index saved to -> /content/drive/MyDrive/MiFO/data/processed/corpus_index.csv


In [9]:
# E1 — Title embeddings for the locked corpus
import subprocess, sys
# Install sentence-transformers if not already installed
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])

import pandas as pd, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

W = Path("/content/drive/MyDrive/MiFO")
corpus = pd.read_csv(W / "data/processed/corpus_index.csv")
print(f"Loaded locked corpus: {len(corpus)} articles")

# Load lightweight, fast 384-dim embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = model.encode(corpus["title"].fillna("").tolist(), batch_size=256,
                   normalize_embeddings=True, show_progress_bar=True)

np.save("/content/title_emb.npy", emb)
np.save("/content/title_ids.npy", corpus["id"].to_numpy())

# Save directly to Google Drive
np.save(W / "data/processed/title_emb.npy", emb)
np.save(W / "data/processed/title_ids.npy", corpus["id"].to_numpy())
print("\nSaved embeddings shape:", emb.shape)

Loaded locked corpus: 12930 articles


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/51 [00:00<?, ?it/s]


Saved embeddings shape: (12930, 384)


In [10]:
# E2 — Event clustering via k-NN graph connected components
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

W = Path("/content/drive/MyDrive/MiFO")
corpus = pd.read_csv(W / "data/processed/corpus_index.csv").reset_index(drop=True)
emb = np.load(W / "data/processed/title_emb.npy")

# Connect articles with cosine similarity >= 0.55 (distance <= 0.45)
k = 5
nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(emb)
dist, ind = nn.kneighbors(emb)

rows, cols = [], []
for i in range(len(emb)):
    for j, d in zip(ind[i, 1:], dist[i, 1:]):
        if d <= 0.45:  # cosine similarity >= 0.55
            rows += [i, j]; cols += [j, i]

g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(emb),)*2)
n_comp, labels = connected_components(g, directed=False)

corpus["event"] = labels
sizes = corpus.groupby("event").size()
multi = corpus[corpus["event"].isin(sizes[sizes >= 3].index)]

print(f"\n==========================================")
print(f"🎯 EVENT CLUSTERING RESULTS")
print(f"==========================================")
print(f"Total Unique Events Found: {n_comp}")
print(f"Multi-Article Events (>=3 articles): {(sizes >= 3).sum()} covering {len(multi)} articles")
print(f"Single-Article Events: {(sizes == 1).sum()}")

# THE CRITICAL MiFO METRIC:
mix = multi.groupby("event")["label_name"].nunique()
print(f"\nEvents with BOTH Fake & Real Coverage: {(mix == 2).sum()} of {len(mix)} ({(mix == 2).mean():.1%})")
print(f"--> These mixed events form the core universe for Anchor vs. Drift testing!")

print("\nLargest Event Clusters (Size | Fake vs. Real | Sample Titles):")
for ev in sizes.sort_values(ascending=False).head(8).index:
    sub = corpus[corpus.event == ev]
    print(f"\n  [Event #{ev} | Size: {len(sub)} | Breakdown: {dict(sub.label_name.value_counts())}]")
    for t in sub["title"].head(3):
        print(f"    • {str(t)[:90]}")

out_events = W / "data/processed/corpus_events_v1.csv"
corpus.to_csv(out_events, index=False)
print(f"\nSaved event clusters to -> {out_events}")


🎯 EVENT CLUSTERING RESULTS
Total Unique Events Found: 3006
Multi-Article Events (>=3 articles): 167 covering 9835 articles
Single-Article Events: 2583

Events with BOTH Fake & Real Coverage: 43 of 167 (25.7%)
--> These mixed events form the core universe for Anchor vs. Drift testing!

Largest Event Clusters (Size | Fake vs. Real | Sample Titles):

  [Event #0 | Size: 8980 | Breakdown: {'real': np.int64(6603), 'fake': np.int64(2377)}]
    • Blac Chyna talks about Rob Kardashian's alleged revenge porn
    • Celebrity Personal Assistant: Salary
    • Taylor Swift Now Has More YouTube Subscribers Than Rihanna

  [Event #14 | Size: 24 | Breakdown: {'real': np.int64(24)}]
    • Simone Biles cries as she talks about Larry Nassar
    • Chrissy Teigen offers to pay $100,000 fine for McKayla Maroney to speak out against Nassar
    • Gymnast Jordyn Wieber tells Senate panel Nassar began abusing her at age 14

  [Event #23 | Size: 20 | Breakdown: {'real': np.int64(20)}]
    • 'Riverdale' Stars Gu

In [11]:
# E3 — Find the similarity threshold where the giant blob breaks
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

W = Path("/content/drive/MyDrive/MiFO")
corpus = pd.read_csv(W / "data/processed/corpus_index.csv").reset_index(drop=True)
emb = np.load(W / "data/processed/title_emb.npy")

nn = NearestNeighbors(n_neighbors=11, metric="cosine").fit(emb)
dist, ind = nn.kneighbors(emb)

print("=== THRESHOLD SWEEP ===")
for thr in [0.45, 0.35, 0.30, 0.25, 0.20]:     # cosine distance (similarity = 1 - thr)
    rows, cols = [], []
    for i in range(len(emb)):
        for j, d in zip(ind[i, 1:], dist[i, 1:]):
            if d <= thr:
                rows += [i, j]; cols += [j, i]
    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(emb),)*2)
    n, labels = connected_components(g, directed=False)
    sizes = pd.Series(labels).value_counts()
    multi = sizes[sizes >= 3]
    df = pd.DataFrame({"e": labels, "lab": corpus["label_name"]})
    mixed = df[df["e"].isin(multi.index)].groupby("e")["lab"].nunique()
    print(f"sim>={1-thr:.2f}: events={n:5d} | multi(>=3)={len(multi):4d} "
          f"cover={int(multi.sum()):5d} | max_comp={sizes.max():5d} "
          f"| mixed={(mixed==2).sum():4d}")

=== THRESHOLD SWEEP ===
sim>=0.55: events= 2998 | multi(>=3)= 159 cover= 9835 | max_comp= 9071 | mixed=  40
sim>=0.65: events= 7122 | multi(>=3)= 418 cover= 5641 | max_comp= 3203 | mixed= 153
sim>=0.70: events= 8997 | multi(>=3)= 387 cover= 3719 | max_comp= 1000 | mixed= 162
sim>=0.75: events=10519 | multi(>=3)= 303 cover= 2191 | max_comp=  196 | mixed= 136
sim>=0.80: events=11683 | multi(>=3)= 187 cover=  996 | max_comp=   43 | mixed=  94


In [16]:
import pandas as pd
import numpy as np
import zipfile
import io
import urllib.request
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
from scipy.sparse import hstack

W = Path("/content/drive/MyDrive/MiFO")
out_dir = W / "data/processed/step3_liar"
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Download official LIAR dataset zip
col_names = [
    "id", "label_name", "statement", "subject", "speaker", "job_title",
    "state_info", "party_affiliation", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts", "context"
]

urls = [
    "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip",
    "https://github.com/thiagorainmaker77/liar_dataset/raw/master/liar_dataset.zip"
]

data_loaded = False
for url in urls:
    try:
        print(f"Fetching from {url}...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as resp:
            z = zipfile.ZipFile(io.BytesIO(resp.read()))
            train_df = pd.read_csv(z.open("train.tsv"), sep="\t", header=None, names=col_names)
            valid_df = pd.read_csv(z.open("valid.tsv"), sep="\t", header=None, names=col_names)
            test_df  = pd.read_csv(z.open("test.tsv"),  sep="\t", header=None, names=col_names)
            data_loaded = True
            print("Successfully extracted LIAR TSV splits!")
            break
    except Exception as e:
        print(f"Failed from {url}: {e}")

if not data_loaded:
    # Fallback to direct raw GitHub files
    base_raw = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/"
    train_df = pd.read_csv(base_raw + "train.tsv", sep="\t", header=None, names=col_names)
    valid_df = pd.read_csv(base_raw + "valid.tsv", sep="\t", header=None, names=col_names)
    test_df  = pd.read_csv(base_raw + "test.tsv",  sep="\t", header=None, names=col_names)

# Save local CSV copies for persistence
train_df.to_csv(out_dir / "train.csv", index=False)
valid_df.to_csv(out_dir / "valid.csv", index=False)
test_df.to_csv(out_dir / "test.csv", index=False)

# 2. Standard 6-way Label Alignment
LABEL_ORDER = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
LABEL_MAP = {lbl: i for i, lbl in enumerate(LABEL_ORDER)}
REV_MAP = {i: lbl for lbl, i in LABEL_MAP.items()}

# 3. Extract Text TF-IDF features
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words="english")
X_tr_txt = tfidf.fit_transform(train_df["statement"].fillna(""))
X_va_txt = tfidf.transform(valid_df["statement"].fillna(""))
X_te_txt = tfidf.transform(test_df["statement"].fillna(""))

# 4. Extract Speaker Credit History features (Normalized)
hist_cols = ["barely_true_counts", "false_counts", "half_true_counts", "mostly_true_counts", "pants_on_fire_counts"]
def get_hist_feats(df):
    feats = df[hist_cols].fillna(0).values.astype(float)
    sums = feats.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1.0  # Avoid division by zero
    return feats / sums

X_tr_hist = get_hist_feats(train_df)
X_va_hist = get_hist_feats(valid_df)
X_te_hist = get_hist_feats(test_df)

# 5. Combine Text + Speaker History
X_train = hstack([X_tr_txt, X_tr_hist])
X_valid = hstack([X_va_txt, X_va_hist])
X_test  = hstack([X_te_txt, X_te_hist])

y_train = train_df["label_name"].map(LABEL_MAP).values
y_valid = valid_df["label_name"].map(LABEL_MAP).values
y_test  = test_df["label_name"].map(LABEL_MAP).values

# 6. Fit Combined Logistic Regression Model
clf = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

# 7. Predict on Test Set and Save
y_pred = clf.predict(X_test)
test_preds_df = pd.DataFrame({
    "statement": test_df["statement"],
    "speaker": test_df["speaker"],
    "y": [REV_MAP[i] for i in y_test],
    "pred": [REV_MAP[i] for i in y_pred]
})
test_preds_df.to_csv(out_dir / "combined_preds.csv", index=False)

# 8. Display Performance & Confusion Matrix
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"\n==========================================")
print(f"🎯 LIAR Combined Model Test Results")
print(f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
print(f"==========================================\n")

print("=== Combined Model Confusion Matrix (Actual y vs Predicted pred) ===")
cm = pd.crosstab(test_preds_df["y"], test_preds_df["pred"], margins=True)
cols_ordered = [lbl for lbl in LABEL_ORDER if lbl in cm.columns] + (["All"] if "All" in cm.columns else [])
rows_ordered = [lbl for lbl in LABEL_ORDER if lbl in cm.index] + (["All"] if "All" in cm.index else [])
print(cm.reindex(index=rows_ordered, columns=cols_ordered).to_string())

Fetching from https://www.cs.ucsb.edu/~william/data/liar_dataset.zip...
Successfully extracted LIAR TSV splits!

🎯 LIAR Combined Model Test Results
Accuracy: 0.4380 | Macro-F1: 0.4441

=== Combined Model Confusion Matrix (Actual y vs Predicted pred) ===
pred         pants-fire  false  barely-true  half-true  mostly-true  true   All
y                                                                              
pants-fire           61     18            4          7            1     1    92
false                32    113           28         31           23    22   249
barely-true          20     31           88         30           27    16   212
half-true             9     29           31        112           55    29   265
mostly-true           5     19           22         45          112    38   241
true                  8     23           23         27           58    69   208
All                 135    233          196        252          276   175  1267


In [17]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.metrics.pairwise import cosine_similarity

W = Path("/content/drive/MyDrive/MiFO")

# 1. Load locked corpus index and title embeddings
corpus_df = pd.read_csv(W / "data/processed/corpus_index.csv")
title_emb = np.load(W / "data/processed/title_emb.npy")
print(f"Loaded corpus: {len(corpus_df)} articles, embedding shape: {title_emb.shape}")

# Normalize embeddings for fast dot-product cosine similarity
norms = np.linalg.norm(title_emb, axis=1, keepdims=True)
norms[norms == 0] = 1.0
emb_norm = title_emb / norms

# 2. Sweep similarity thresholds
sim_thresholds = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]

print("\n=========================================================================================")
print(f"{'Sim Thresh':>10} | {'Dist Thresh':>11} | {'Total Clusters':>14} | {'Max Size':>9} | {'Clusters >=3':>12} | {'Mixed (Fake+Real)':>17}")
print("=========================================================================================")

sweep_results = []
batch_size = 2000
N = len(corpus_df)

for sim_thresh in sim_thresholds:
    # Build sparse adjacency matrix in batches to stay memory-efficient
    rows, cols = [], []
    for i in range(0, N, batch_size):
        end_i = min(i + batch_size, N)
        sim_batch = np.dot(emb_norm[i:end_i], emb_norm.T)
        
        # Mask out self-loops and values below threshold
        r, c = np.where(sim_batch >= sim_thresh)
        # Shift row indices to global index
        r_global = r + i
        # Keep upper triangle to avoid duplicate undirected edges
        valid = r_global < c
        rows.extend(r_global[valid])
        cols.extend(c[valid])
        
    data = np.ones(len(rows), dtype=bool)
    adj = csr_matrix((data, (rows, cols)), shape=(N, N))
    # Make symmetric for connected components
    adj = adj + adj.T
    
    n_components, labels = connected_components(csgraph=adj, directed=False)
    
    # Analyze cluster distributions
    counts = pd.Series(labels).value_counts()
    max_size = counts.iloc[0]
    clusters_ge_3 = (counts >= 3).sum()
    
    # Count mixed clusters
    df_temp = pd.DataFrame({"label": labels, "truth": corpus_df["label_name"]})
    grouped = df_temp.groupby("label")["truth"].nunique()
    multi_labels = counts[counts >= 3].index
    mixed_count = (grouped.loc[multi_labels] > 1).sum()
    
    sweep_results.append({
        "sim_threshold": sim_thresh,
        "dist_threshold": round(1.0 - sim_thresh, 2),
        "total_clusters": n_components,
        "max_size": max_size,
        "clusters_ge_3": clusters_ge_3,
        "mixed_clusters": mixed_count
    })
    
    print(f"{sim_thresh:>10.2f} | {1.0 - sim_thresh:>11.2f} | {n_components:>14} | {max_size:>9} | {clusters_ge_3:>12} | {mixed_count:>17}")

print("=========================================================================================")

Loaded corpus: 12930 articles, embedding shape: (12930, 384)

Sim Thresh | Dist Thresh | Total Clusters |  Max Size | Clusters >=3 | Mixed (Fake+Real)
      0.55 |        0.45 |           2998 |      9071 |          159 |                40
      0.60 |        0.40 |           5011 |      5788 |          355 |               113
      0.65 |        0.35 |           7122 |      3203 |          418 |               153
      0.70 |        0.30 |           8997 |      1000 |          387 |               162
      0.75 |        0.25 |          10519 |       196 |          303 |               136
      0.80 |        0.20 |          11683 |        43 |          187 |                94
      0.85 |        0.15 |          12315 |        11 |           92 |                51
